# IMPORT LIBRARIES

In [ ]:
### DATA HANDLING ###
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as ss
from scipy.stats import pearsonr
from scipy.fftpack import dct
import shap

### SK LEARN ###
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.model_selection import GroupKFold, ShuffleSplit, StratifiedShuffleSplit, TimeSeriesSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KernelDensity
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression

### Neural Net ###
import sklearn.linear_model
import sklearn.neural_network
import sklearn.model_selection

### BOOST ###
import xgboost
from lightgbm import LGBMClassifier
# from catboost import CatBoostClassifier
 
import warnings
warnings.filterwarnings("ignore")

# USER FUNCTIONS
Data Analysis, Cleaning, Feature Engineering, Modelling and Evaluation

## DATA HANDLING AND FEATURE GENERATION

In [ ]:
### Cacluate descriptors of raw data: Amplitude & Phase
def time_stats(x, prefix):
    return {
        f"{prefix}_mean": np.mean(x, axis=1),
        f"{prefix}_std":  np.std(x, axis=1),
        f"{prefix}_min":  np.min(x, axis=1),
        f"{prefix}_max":  np.max(x, axis=1),
        f"{prefix}_var":  np.var(x, axis=1),
    }

### Calculate rolling values ###
def rolling_smooth(df, feature_cols, window=5):
    df = df.copy()
    df = df.sort_values("timestamp")

    for col in feature_cols:
        df[col] = (
            df[col]
            .rolling(window=window, min_periods=1)
            .mean()
        )

    return df

### Feature Selection by Variance ###
def fit_near_zero_variance(X_train: pd.DataFrame, var_thresh=1e-6):
    variances = X_train.var(axis=0)
    keep_cols = variances[variances > var_thresh].index.tolist()
    drop_cols = variances[variances <= var_thresh].index.tolist()
    return keep_cols, drop_cols

### Amplitude Normalization ###
def normalize_per_sample(amp):
    norm = np.linalg.norm(amp, axis=1, keepdims=True)
    return amp / (norm + 1e-8)

### Discrete Cos Transformation ###
def compute_dct_features(A, n_keep):
    C = dct(A, type=2, axis=1, norm="ortho")
    return C[:, :n_keep]

def smooth_probs(p,alpha):
    p_new = p.copy()
    for i in range(1, p.shape[1]-1):
        p_new[:, i] = (2*alpha)*p[:, i] + alpha*p[:, i-1] + alpha*p[:, i+1]
    return p_new

### PCA on select columns of each antenna ###
def pca_feature_selection(df_train, df_test, f_type, ant, comp):
    pca_cols = list(df_train.columns[(df_train.columns.str.startswith(f_type)) & (df_train.columns.str.endswith(ant))])
    
    ### Baseline Model: RF ###
    pca = Pipeline([
        ('center', StandardScaler()),
            ('clf', PCA(n_components=comp)
        )
    ])

    df_pca = pd.DataFrame(pca.fit_transform(df_train))
    df_pca.columns = [f_type+str(i)+str(ant) for i in df_pca.columns]
    df_train = df_train.drop(columns=pca_cols)
    df_pca.index = df_train.index

    df_pca_val = pd.DataFrame(pca.transform(df_test))
    df_pca_val.columns = [f_type+str(i)+str(ant) for i in df_pca_val.columns]
    df_test = df_test.drop(columns=pca_cols)
    df_pca_val.index = df_test.index

    df_train = pd.concat([df_train, df_pca],axis=1)
    df_test = pd.concat([df_test, df_pca_val],axis=1)
    
    return df_train, df_test

# ### MDA on selected features ###
# def MDA_feature_selection(df_train_x, df_train_y, df_test, f_type, ant, k=9):
#     mda_cols = list(df_train_x.columns[(df_train_x.columns.str.startswith(f_type)) & (df_train_x.columns.str.endswith(ant))])
    
#     mda = LinearDiscriminantAnalysis(n_components=k)

#     df_mda = pd.DataFrame(mda.fit_transform(df_train_x,df_train_y))
#     df_mda.columns = [f_type+str(i)+str(ant) for i in df_mda.columns]
#     df_train = df_train_x.drop(columns=mda_cols)
#     df_mda.index = df_train.index

#     df_mda_val = pd.DataFrame(mda.transform(df_test))
#     df_mda_val.columns = [f_type+str(i)+str(ant) for i in df_mda_val.columns]
#     df_test = df_test.drop(columns=mda_cols)
#     df_mda_val.index = df_test.index

#     df_train = pd.concat([df_train, df_mda],axis=1)
#     df_test = pd.concat([df_test, df_mda_val],axis=1)
    
#     return df_train, df_test

In [ ]:
### Pau Ta outlier treatment: clip outliers beyond mean +/- 3*std Dev ###
def pauta_fit(amp_train, k=3.0):
    fit_cols = list(amp_train.columns[(amp_train.columns.str.startswith('amp')) & 
                                      ((amp_train.columns.str.endswith("_1"))|(amp_train.columns.str.endswith("_2")))])

    # fit_cols = list(amp_train.columns[(amp_train.columns.str.startswith('dct_amp'))]) 
    # print(fit_cols)
    mu = amp_train[fit_cols].mean(axis=0)
    sigma = amp_train[fit_cols].std(axis=0)

    lower = mu - k * sigma
    upper = mu + k * sigma

    return {"k": k, "mu": mu, "sigma": sigma, "lower": lower, "upper": upper}

def pauta_transform_clip(amp, params):
    fit_cols = list(amp.columns[(amp.columns.str.startswith('amp')) & 
                                      ((amp.columns.str.endswith("_1"))|(amp.columns.str.endswith("_2")))])
    lower = params["lower"]
    upper = params["upper"]

    amp_clean = amp[fit_cols].clip(lower=lower, upper=upper, axis=1)
    amp = amp.drop(fit_cols, axis=1)
    amp = pd.concat([amp,amp_clean],axis=1)

    return amp

In [ ]:
### Plot Amplitude & Phase Distribution ###
def plot_amplitude_per_subcarrier(amplitude_data, labels):
    df = pd.DataFrame(amplitude_data)
    df['position'] = labels
    
    # Melt
    df_melt = df.melt(id_vars='position', var_name='subcarrier', value_name='amplitude').reset_index(drop=True)
    
    plt.figure(figsize=(7, 3))
    sns.lineplot(data=df_melt, x='subcarrier', y='amplitude', hue='position', palette='viridis')
    plt.title('Mean Amplitude per Subcarrier across 10 Positions')
    plt.xlabel('Subcarrier Index')
    plt.ylabel('Amplitude (Magnitude)')
    plt.legend(title='Position', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_amplitude_distribution(amplitude_data, labels):

    # Calculate mean amplitude per packet
    mean_amp_per_packet = np.mean(amplitude_data, axis=1)
    
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=labels, y=mean_amp_per_packet, palette='Set3')
    plt.title('Distribution of Average Amplitude by Position')
    plt.xlabel('Position (Class)')
    plt.ylabel('Average Amplitude')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

### Compare Confused pair of classes ###
def compare_class_pairs(cm,y):
    cm_copy = cm.copy()
    np.fill_diagonal(cm_copy, 0)

    pairs = []
    classes = np.sort(y.unique())

    for i in range(len(classes)):
        for j in range(len(classes)):
            if i != j and cm_copy[i, j] > 0:
                pairs.append((classes[i], classes[j], cm_copy[i, j]))

    pairs_sorted = sorted(pairs, key=lambda x: x[2], reverse=True)

    print("Top 2 confusions (true, predicted, misclassifications):")
    for p in pairs_sorted[:2]:
        print(p)
    return pairs_sorted

### Plot ROC for multiclass problem ###
def multiclass_roc(y_true, y_proba, k, classes=None, plot=True, grid=True):
    title="Multiclass ROC (One-vs-Rest)"
    figsize=(7, 6)
    
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)

    if classes is None:
        classes = np.unique(y_true)
    classes = np.asarray(classes)

    # Binarize labels
    y_true_bin = label_binarize(y_true, classes=classes)

    if y_proba.shape[1] != len(classes):
        raise ValueError(
            f"y_proba has {y_proba.shape[1]} columns but classes has {len(classes)} values. "
            "Make sure columns of y_proba match the class order."
        )

    fpr, tpr, thresholds, auc_per_class = {}, {}, {}, {}

    # Per-class ROC + AUC
    for i, cls in enumerate(classes):
        fpr[i], tpr[i], thresholds[i] = roc_curve(y_true_bin[:, i], y_proba[:, i])
        auc_per_class[i] = auc(fpr[i], tpr[i])

    micro_auc = roc_auc_score(y_true_bin, y_proba, average="micro")
    macro_auc = roc_auc_score(y_true_bin, y_proba, average="macro")

    # bottom-k classes by AUC
    sorted_idxs = sorted(auc_per_class.keys(), key=lambda i: auc_per_class[i])
    bottom_k_idxs = sorted_idxs[: min(k, len(sorted_idxs))]

    if plot:
        plt.figure(figsize=figsize)

        for i in bottom_k_idxs:
            plt.plot(
                fpr[i],
                tpr[i],
                label=f"Class {classes[i]} (AUC={auc_per_class[i]:.3f})"
            )

        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"{title}\nMicro AUC={micro_auc:.3f}, Macro AUC={macro_auc:.3f}")
        plt.legend()
        if grid:
            plt.grid(True)
        plt.show()

    return {
        "classes": classes,
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "auc_per_class": auc_per_class,
        "micro_auc": micro_auc,
        "macro_auc": macro_auc,
        "bottom_k_class_indices": bottom_k_idxs,
        "bottom_k_classes": classes[bottom_k_idxs],
    }

def detrend_feature(df, features, time_col='time_rel_s'):
    df = df.copy()
    t = df.loc[:,time_col].values.reshape(-1,1)
    for f in features:
        y = df.loc[:,f].values        
        model = LinearRegression()
        model.fit(t, y)
        trend = model.predict(t)        
        residuals = y - trend
        
        df[f'{f}_detrended'] = residuals
        # df[f'{f}_dt_roll'] = df[f'{f}_detrended'].rolling(window=5, min_periods=1).mean()

    return df

## FUNCTIONS FOR TRAIN & TEST
Generate calculated fields such as Amplitude, DCT on Amplitude, PSD on Amplitude, phase, phase slope & delay, AOA/
Descriptors: Mean, Std Dif, Difference & Roll features/

In [ ]:
def generate_features(df_csi):
    eps = 1e-6
    ### IQ Columns for amplitude calculation ###
    I1 = df_csi.filter(regex=r"^I.*_1$").sort_index(axis=1)
    Q1 = df_csi.filter(regex=r"^Q.*_1$").sort_index(axis=1)
    I1_vals = I1.to_numpy()
    Q1_vals = Q1.to_numpy()

    I2 = df_csi.filter(regex=r"^I.*_2$").sort_index(axis=1)
    Q2 = df_csi.filter(regex=r"^Q.*_2$").sort_index(axis=1)
    I2_vals = I2.to_numpy()
    Q2_vals = Q2.to_numpy()

    # Calculate amplitude, mean energy & phase
    amplitude_1 = np.sqrt(I1_vals**2 + Q1_vals**2)
    amplitude_2 = np.sqrt(I2_vals**2 + Q2_vals**2)

    ## Compute Path Loss
    # pathloss1 = -np.log(np.mean(amplitude_1,axis=1))
    # pathloss2 = -np.log(np.mean(amplitude_2,axis=1))

    ## Normalize Amplitude across subcarriers ##
    amplitude_1 = normalize_per_sample(amplitude_1)
    amplitude_2 = normalize_per_sample(amplitude_2)

    ### PSD on amplitude: analyse shape and angle ###
    FFT_a1 = np.fft.rfft(amplitude_1, axis=1)
    FFT_a2 = np.fft.rfft(amplitude_2, axis=1)
    psd1 = (np.abs(FFT_a1) ** 2)
    psd2 = (np.abs(FFT_a2) ** 2)
    ## Max on PSD as feature ##
    psd1 = psd1.max(axis=1)
    psd2 = psd2.max(axis=1)

    # ## Kurtosis Calculation ##
    # m2 = np.mean(amplitude_1**2, axis=1) + eps
    # m4 = np.mean(amplitude_2**4, axis=1)
    # kurt = (m4 / (m2**2)) + 3

    ## Calculate Phase ##
    phase1 = np.arctan2(Q1_vals, I1_vals)
    phase2 = np.arctan2(Q2_vals, I2_vals)
    phase_diff = np.unwrap(phase1 - phase2, axis=1)
    phase1 = np.unwrap(phase1, axis=1)
    phase2 = np.unwrap(phase2, axis=1)

    ## phase features: sin & Cos on the wrapped phase difference ##
    H1 = I1_vals + 1j * Q1_vals
    H2 = I2_vals + 1j * Q2_vals
    complex_r = H1 / (H2 + eps)
    dphi = np.unwrap(np.angle(complex_r), axis=1)           
    A1 = np.abs(H1)
    A2 = np.abs(H2)
    w = (A1 + A2) + eps
    sin_mean = np.sum(w * np.sin(np.angle(complex_r)), axis=1) / np.sum(w, axis=1)
    cos_mean = np.sum(w * np.cos(np.angle(complex_r)), axis=1) / np.sum(w, axis=1)
    
    ## AOA Calculation
    c = 3e8 ## speed of transmission
    lam1 = 6.25 ## 2.5GHz 1st assumed wavelength
    lam2 = 2.9 ## 5GHz 2nd assumed wavelength
    antenna_spacing_m = 6 ## Spacing between antennas

    # wrap dphi to [-pi, pi] ##
    dphi_wrapped = (phase_diff + np.pi) % (2*np.pi) - np.pi

    # Derive AoA from phase values ##
    aoa_arg1 = (dphi_wrapped * lam1) / (2 * np.pi * antenna_spacing_m)
    aoa_arg2 = (dphi_wrapped * lam2) / (2 * np.pi * antenna_spacing_m)
    aoa_arg1 = np.clip(aoa_arg1, -1.0, 1.0)
    aoa_arg2 = np.clip(aoa_arg2, -1.0, 1.0)
    aoa_rad1 = np.arcsin(aoa_arg1)
    aoa_rad2 = np.arcsin(aoa_arg2)
    aoa_packet1 = np.mean(aoa_rad1, axis=1)
    aoa_packet2 = np.mean(aoa_rad2, axis=1)

    ## calculate phase slope of phase & delay ##
    k1 = np.arange(phase1.shape[1])
    phase_slope1 = np.array([np.polyfit(k1, phase1[t], 1)[0] for t in range(phase1.shape[0])])
    k2 = np.arange(phase2.shape[1])
    phase_slope2 = np.array([np.polyfit(k2, phase2[t], 1)[0] for t in range(phase2.shape[0])])

    #######################################################################################################################

    ### ADD Calculated Fields ###
    delta_f_hz=312_500
    phase_slope1 = pd.DataFrame(
        phase_slope1,
        columns=[f"phase_slope1"],
        index=df_csi.index
    )
    phase_slope1['tau_delay1'] = -(phase_slope1['phase_slope1'] / delta_f_hz) / (2*np.pi)

    phase_slope2 = pd.DataFrame(
        phase_slope2,
        columns=[f"phase_slope2"],
        index=df_csi.index
    )
    phase_slope2['tau_delay2'] = -(phase_slope2['phase_slope2'] / delta_f_hz) / (2*np.pi)

    amp_df1 = pd.DataFrame(
        amplitude_1,
        columns=[f"amplitude_{c}" for c in I1.columns],
        index=df_csi.index
    )
    amp_df1['amp1_mean_e'] = amp_df1.apply(lambda x: np.mean(x**2), axis=1)

    amp_df2 = pd.DataFrame(
        amplitude_2,
        columns=[f"amplitude_{c}" for c in I2.columns],
        index=df_csi.index
    )
    amp_df2['amp2_mean_e'] = amp_df2.apply(lambda x: np.mean(x**2), axis=1)

    amp_df2['aoa_calculated1'] = aoa_packet1
    amp_df2['aoa_calculated2'] = aoa_packet2
    amp_df2['aoa_calculated_diff'] = amp_df2['aoa_calculated1'] - amp_df2['aoa_calculated2']
    # amp_df2['pathloss1'] = pathloss1
    # amp_df2['pathloss2'] = pathloss2
    # amp_df2['pathloss_diff'] = amp_df2['pathloss1'] - amp_df2['pathloss2']

    phase_df1 = pd.DataFrame(
        phase1,
        columns=[f"phase_{c}" for c in I1.columns],
        index=df_csi.index
    )

    phase_df2 = pd.DataFrame(
        phase2,
        columns=[f"phase_{c}" for c in I2.columns],
        index=df_csi.index
    )

    # phase_diff = pd.DataFrame(
    #     dphi,
    #     columns=[f"phase_diff_{c}" for c in range(dphi.shape[1])],
    #     index=df_csi.index
    # )

    ### Discrete Cosine Transform: Denoising high frequency Amplitude ###
    K=13 ## Cosine coefficients kept: Tuned using trial & error
    dct1 = compute_dct_features(amplitude_1, n_keep=K)
    dct2 = compute_dct_features(amplitude_2, n_keep=K)
    for i in range(K):
        amp_df1[f"dct_amp1_{i}"] = dct1[:, i]
        amp_df2[f"dct_amp2_{i}"] = dct2[:, i]

    ## DCT: polynomial transformations 
    dct_cols1 = [c for c in amp_df1.columns if c.startswith("dct_amp1_")]
    dct_cols2 = [c for c in amp_df2.columns if c.startswith("dct_amp2_")]
    amp_df1["dct_energy1"] = (amp_df1[dct_cols1]**2).sum(axis=1)
    amp_df2["dct_energy2"] = (amp_df2[dct_cols2]**2).sum(axis=1)

    amp_df2["phase_diff_s"] = sin_mean
    amp_df2["phase_diff_c"] = cos_mean
    amp_df2["psd1"] = psd1
    amp_df2["psd2"] = psd2
    amp_df2["psd_max_diff"] = amp_df2["psd1"] - amp_df2["psd2"]
    amp_df2["psd_max_ratio"] = amp_df2["psd1"] / (amp_df2["psd2"] + eps)
    # amp_df2["kurt_coeff"] = kurt

    #######################################################################################################################

    ### Concatinate all CSI fingerprint features 
    df_csi1 = pd.concat([df_csi, amp_df1, amp_df2, phase_slope1, phase_slope2], axis=1)

    # ### Summarize amplitude fingerprint as descriptors
    # amplitude1_stats = pd.DataFrame(time_stats(df_csi1.filter(regex=r"^amp.*_1$"),'amplitude_1'))
    # amplitude2_stats = pd.DataFrame(time_stats(df_csi1.filter(regex=r"^amp.*_2$"),'amplitude_2'))

    ## Drop the raw fields: I, Q, Amplitude, Phase
    # df_csi2 = pd.concat([df_csi1, amplitude1_stats, amplitude2_stats], axis=1)
    df_csi2 = df_csi1.drop(list(I1.columns) + list(I2.columns) + list(Q1.columns) + list(Q2.columns),axis=1)
    drop_cols_amp = list(df_csi1.filter(regex=r"^amp.*_1$").columns) + list(df_csi1.filter(regex=r"^amp.*_2$").columns)
    # drop_cols_amp = list(dct_cols1) + list(dct_cols2)
    df_csi2 = df_csi2.drop(columns=drop_cols_amp)

    #######################################################################################################################

    ## Add Timestamp related features: Cyclicity accounted using sin & cos transformation on time ##
    df_csi2['time_s'] = (df_csi2['timestamp'] / 1_000_000)   # seconds
    df_csi2['time_rad'] = (2 * np.pi * df_csi2['time_s']) % (2*np.pi)
    df_csi2['time_sin'] = np.sin(df_csi2['time_rad'])
    df_csi2['time_cos'] = np.cos(df_csi2['time_rad'])
    df_csi2['time_rel_s'] = df_csi2['time_s'] - df_csi2['time_s'].iloc[0]
    df_csi2['delta_t_s'] = df_csi2['time_rel_s'].diff().fillna(0)
    
    ## Add Rssi related cross antenna features: sum, mean, roll ##
    df_csi2['rssi_diff'] = df_csi2['rssi1'] - df_csi2['rssi2']
    df_csi2['rssi_abs_diff'] = abs(df_csi2['rssi1'] - df_csi2['rssi2'])
    df_csi2['rssi_sum'] = df_csi2['rssi1'] + df_csi2['rssi2']
    df_csi2['rssi_mean'] = (df_csi2['rssi1'] + df_csi2['rssi2'])/2
    df_csi2['rssi_mean_sq'] = df_csi2['rssi_mean']**2
    df_csi2['rssi1_rolling'] = df_csi2['rssi1'].rolling(window=3, min_periods=1).mean()
    df_csi2['rssi2_rolling'] = df_csi2['rssi2'].rolling(window=3, min_periods=1).mean()
    df_csi2['rssi_mean_rolling'] = df_csi2['rssi_mean'].rolling(window=3, min_periods=1).mean()

    ## AOA Feature Transforms: sin & cos transformations, interactions with rssi ##
    df_csi2['aoa_rad'] = np.deg2rad(df_csi2['aoa'])
    df_csi2['aoa_sin'] = np.sin(df_csi2['aoa_rad'])
    df_csi2['aoa_cos'] = np.cos(df_csi2['aoa_rad'])
    df_csi2['aoa_rssi1'] = df_csi2['aoa']*df_csi2['rssi1']
    df_csi2['aoa_rssi2'] = df_csi2['aoa']*df_csi2['rssi2']
    df_csi2['aoa_rssi_mean'] = df_csi2['aoa']*df_csi2['rssi_mean']
    # df_csi2['aoa_rssi_diff'] = df_csi2['aoa_rssi1']-df_csi2['aoa_rssi2']
    df_csi2['seq_ctrl_diff'] = df_csi2['seq_ctrl'].diff().fillna(0)

    ## Amplitude Feature Transforms ##
    eps = 1e-6
    # df_csi2['amp_mean_ratio'] = df_csi2['amplitude_1_mean']/(df_csi2['amplitude_2_mean']+eps)
    # df_csi2['amp_std_diff'] = df_csi2['amplitude_1_std']-df_csi2['amplitude_2_std']
    # df_csi2['amp_std_m_diff'] = df_csi2['amplitude_1_max']-df_csi2['amplitude_2_min']
    df_csi2['dct_energy_diff'] = df_csi2['dct_energy1']-df_csi2['dct_energy2']
    # df_csi2['dct_energy_mean'] = (df_csi2['dct_energy1']+df_csi2['dct_energy2'])/2
    df_csi2['p_slope_mean'] = (df_csi2['phase_slope1']+df_csi2['phase_slope2'])/2
    
    ## DCT features: Cross antenna ratios & differences ##
    df_csi2["dct_ratio_4_5"] = df_csi2["dct_amp2_4"]/(abs(df_csi2["dct_amp2_5"]) + eps)
    df_csi2["dct_ratio_3_4"] = df_csi2["dct_amp1_3"]/(abs(df_csi2["dct_amp1_4"]) + eps)
    # df_csi2["dct_ratio_6_12"] = df_csi2["dct_amp1_12"]/(abs(df_csi2["dct_amp1_6"]) + eps)
    df_csi2["dct4_mean"] = (df_csi2["dct_amp1_4"] + df_csi2["dct_amp2_4"])/2
    df_csi2["dct5_mean"] = (df_csi2["dct_amp1_5"] + df_csi2["dct_amp2_5"])/2
    df_csi2["dct6_mean"] = (df_csi2["dct_amp1_6"] + df_csi2["dct_amp2_6"])/2
    # df_csi2["dct2_mean"] = (df_csi2["dct_amp1_2"] + df_csi2["dct_amp2_2"])/2
    # df_csi2["dct1_mean"] = (df_csi2["dct_amp1_1"] + df_csi2["dct_amp2_1"])/2
    df_csi2["dct5_diff"] = (df_csi2["dct_amp1_5"] - df_csi2["dct_amp2_5"])
    # df_csi2["dct6_diff"] = (df_csi2["dct_amp1_6"] - df_csi2["dct_amp2_6"])

    ## Detrend and Capture periodicity ##
    # df_csi2 = detrend_feature(df_csi2, ["rssi_sum"], time_col='time_rel_s')
    # df_csi2['time_rel_s_lag1'] = df_csi2['time_rel_s'].shift(1)
    # df_csi2['time_rel_s_lag2'] = df_csi2['time_rel_s'].shift(2)
    df_csi2 = df_csi2.fillna(0)

    #######################################################################################################################

    ## Feature - Target split
    drop_cols = ["position", "timestamp", "time_s",'time_rad','seq_ctrl','aoa_rad','aoa_calculated1','aoa_calculated_diff']
    X = df_csi2.drop(columns=drop_cols)
    y = df_csi2["position"]
    
    return X, y, phase_df1, phase_df2, amplitude_1, amplitude_2

### Model Run Function ###
Baseline: Random Forest - high overfitting on training data due to high dimensionality\
XGB: reduces bias using boosting & manages variance through regularization

In [ ]:
def model_run(df_csi_train,df_csi_test,type_,threshold,t_seed):

    ## Generate the required features separately on train & test data 
    X_train, y_train, phase_df1_train, phase_df2_train, amp1_train, amp2_train = generate_features(df_csi_train)
    X_val, y_val, phase_df1_test, phase_df2_test, amp1_test, amp2_test = generate_features(df_csi_test)

    train_id = X_train[["ID"]]
    test_id = X_val[["ID"]]
    X_train = X_train.drop(columns=["ID"])
    X_val = X_val.drop(columns=["ID"])
    
    xgb_params = {
            "objective":"multi:softprob", 
            "num_class":y_train.nunique(),
            "n_estimators":600,
            "max_depth":5,
            "tree_method": "hist",
            "learning_rate":0.05,
            "subsample":0.8,
            "colsample_bytree":0.8,
            "random_state":42,
            "eval_metric":"mlogloss",
            "reg_lambda":1,
            "n_jobs":-1,
            "class_weight":"balanced",
            "min_child_samples":5
        }
    
    ### Initialize model Pipeline: Scaling & XGBoost ###
    model =  xgboost.XGBClassifier(**xgb_params)

    # ### Baseline Model: RF ###
    # model = Pipeline([
    #     ('center', MinMaxScaler()),
    #         ('clf', SVC(kernel="rbf")
    #     )
    # ])

    ## prune features: Dynamic selection ##
    # keep_cols, drop_cols = fit_near_zero_variance(X_train, var_thresh=1e-4)
    # model.fit(X_train, y_train)
    # explainer = shap.TreeExplainer(model)
    # shap_vals = explainer.shap_values(X_train.sample(min(3000, len(X_train)),random_state=t_seed))
    # shap_abs = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
    # imp = shap_abs.mean(axis=1)
    # imp_df = pd.DataFrame({
    #     "feature": X_train.columns,
    #     "importance": imp
    # }).sort_values("importance", ascending=False)
    # keep_cols = imp_df.head(threshold)["feature"].tolist()

    ## SHAP selected Features with best CV ##
    keep_cols = ['time_rel_s', 'rssi_sum', 'rssi2', 'rssi1', 'dct_amp2_3', 'dct_amp1_4', 'dct_amp2_4', 'dct_energy1', 
                'dct_amp2_7', 'dct_amp2_2', 'dct_amp1_11', 'dct_amp2_5', 'dct_amp2_9', 'dct_amp1_7', 'dct5_mean', 
                'dct_ratio_4_5', 'dct_amp1_6', 'dct_amp1_12', 'dct_amp1_1', 'dct_amp2_6', 'dct4_mean', 'rssi_diff', 
                'rssi_mean', 'dct_amp1_2', 'dct_amp2_8', 'dct_amp1_10', 'dct_amp2_11', 'dct_energy2', 'dct_amp1_8', 
                'dct_amp1_5', 'dct_amp1_0', 'dct_amp1_9', 'dct6_mean', 'dct_ratio_3_4', 'dct_amp2_1', 'dct_amp1_3', 
                'dct_amp2_0', 'dct_amp2_10', 'dct5_diff', 'time_cos', 'dct_amp2_12', 'rssi_mean_rolling', 'delta_t_s', 
                'time_sin', 'dct_energy_diff', 'psd_max_diff', 'rssi2_rolling', 'rssi1_rolling', 'seq_ctrl_diff', 
                'psd_max_ratio', 'phase_slope1', 'phase_diff_c', 'rssi_abs_diff', 'phase_diff_s', 'aoa_calculated2', 'psd1', 
                'rssi_mean_sq', 'psd2', 'phase_slope2', 'p_slope_mean', 'tau_delay1', 'aoa_cos', 'tau_delay2']
    X_train = X_train[keep_cols]
    X_val  = X_val[keep_cols]

    # ## PauTa Outlier Treatment
    # params = pauta_fit(X_train, k=3.0)
    # X_train = pauta_transform_clip(X_train, params)
    # X_val  = pauta_transform_clip(X_val, params)

    ### PCA on amplitude ###
    # X_train, X_val = pca_feature_selection(X_train, X_val, "amp", "_1", 0.9)
    # X_train, X_val = pca_feature_selection(X_train, X_val, "amp", "_2", 0.9)
    
    print(list(X_train.columns))
    print(len(X_train.columns))

    ### Model Training & Prediction ###
    model =  xgboost.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_train)
    confidence = proba.max(axis=1)
    sample_weights = 1 + (1 - confidence)
    # model =  xgboost.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train,sample_weight=sample_weights)   
    # preds = model.predict(X_val)

    ## Prob Sharpening ##
    y_proba = model.predict_proba(X_val)
    y_proba = y_proba ** 1.5
    y_proba = y_proba / y_proba.sum(axis=1, keepdims=True)

    ## Smoothen adjacent locations ##
    # y_proba = smooth_probs(y_proba,0.2)
    preds = np.argmax(y_proba, axis=1)

    # print(preds)
    df_pred = pd.DataFrame()
    df_pred['ID'] = test_id
    df_pred['position'] = preds

    X_train['ID'] = train_id
    X_train['y'] = y_train
    X_val['ID'] = test_id

    ### EVALUATION METRICS ###
    if type_ == "testing":
        f1 = f1_score(y_val, preds, average="macro")
        acc = accuracy_score(y_val, preds)
        prec = precision_score(y_val, preds, average="macro")
        rec = recall_score(y_val, preds, average="macro")
        metric_df = pd.DataFrame({'accuracy':[acc],"F1":[f1],"precision":prec,"recall":rec})

        ### Enable if Visualizing/Validating ###
        ## Compare conflicting pairs ##
        # cm_pairs = compare_class_pairs(confusion_matrix(y_val, preds),y_val)
        
        ## ROC ##
        # results = multiclass_roc(y_val, y_proba, 3)
        # print(results["micro_auc"], results["macro_auc"])
        
        ### Print Metrics: Classification score and CM ###
        # print("TRAINING\n" + classification_report(y_train, pred_train))
        print("\nTESTING\n" + classification_report(y_val, preds))
        # print(confusion_matrix(y_val, preds))

        print(f"F1 = {f1:.4f}")
    else:
        print("Blind Prediction on Test Data")
        metric_df=0

    return metric_df, df_pred, X_val, X_train

# LOAD DATASET

Channel State Information Dataset

In [ ]:
### CONFIGURATION ###
basepath = "..\\data\\" ## adjust location
paths = {
    "train_dataset":basepath + "train.csv",
    "test_dataset":basepath + "test_nolabels.csv",
    "output_dataset":basepath + "test_prediction.csv"
}

## Load Train Data ##
df_csi = pd.read_csv(paths["train_dataset"], index_col=0)
indexes = df_csi.index
df_csi['ID'] = indexes
df_csi.head()

## Sort by Timestamp: capture time related structures in data
df_csi = df_csi.sort_values("timestamp").reset_index(drop=True)
df_csi

# EXPLORATORY DATA EXPLORATION

### Basic & Sanity Checks
Missing, Nulls, Mean, Feature Data type, Outliers

In [ ]:
### Descriptors ###
print(df_csi.describe())
print(df_csi.info())

### TSNE analysis: Check separability of Raw data 
Raw features have low local separability: based on TSNE there is high overlap in distributions of different classes. High Noise in raw IQ complex values.

In [ ]:
### Raw T-Sne data Check for separability ### 
X = df_csi.drop(['position','timestamp'], axis=1).to_numpy()
y = df_csi['position'].to_numpy()
print(X.shape)

## Scale Features ##
X_scaled = MinMaxScaler().fit_transform(X)

## Initialize and fit SNE ##
tsne = TSNE(
    n_components=2,
    perplexity=50,
    learning_rate=50,
    random_state=2
)

X_tsne = tsne.fit_transform(X_scaled)
plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, cmap='tab10', s=5)
plt.title("t-SNE raw: before denoising")
plt.show()

### Separability and distribution across RSSI & AOA
RSSI shows strong separability of positions on distance: Closer positions have higher mean rssi relative to further classes\
AOA has similar means across angles & distance: Weak separator of angles independently

In [ ]:
# scatter plot shuffled data
sns.pairplot(df_csi[["rssi1","rssi2","aoa","position"]],hue='position');
plt.draw()

In [ ]:
for feature in ["rssi1","rssi2","aoa"]:
    sns.histplot(data=df_csi[["rssi1","rssi2",'aoa',"position"]], x=feature, hue="position",kde=True, bins=30, multiple="stack")
    plt.show()

In [ ]:
## RSSI: STRONG distance Separator but weak anglular separator
fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey=False)

# AOA boxplot
df_csi.boxplot(column="aoa", by="position", ax=axes[0])
axes[0].set_title("AOA by Position")

# RSSI1 boxplot
df_csi.boxplot(column="rssi1", by="position", ax=axes[1])
axes[1].set_title("RSSI1 by Position")

# RSSI2 boxplot
df_csi.boxplot(column="rssi2", by="position", ax=axes[2])
axes[2].set_title("RSSI2 by Position")

plt.suptitle("Boxplot Analysis of RSSI & AOA")
plt.tight_layout()
plt.show()

### Class Imbalance Check
Classes are balanced across positions: both macro F1 & error metrics are feasible for evaluation

In [ ]:
# count classes
class_counts = df_csi['position'].value_counts()

# plot
class_counts.plot(kind='bar')
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Count of Samples by Class')
plt.show()

### Timestamp Periodicity Check

In [ ]:
df_csi_ = df_csi.sort_values(by=['timestamp'],ascending=True) 
df_csi_['delta_t_us'] = df_csi_['timestamp'].diff()
plt.plot(df_csi_.reset_index().head(1000).index, df_csi_.head(1000)['delta_t_us'])
plt.grid()

In [ ]:
## Time Delta Distribution: Peak time interval suggests strong tendency to mean periodicity ##
mean_dt = df_csi_['delta_t_us'].mean()
std_dt = df_csi_['delta_t_us'].std()
cv = std_dt / mean_dt
print(cv)

plt.hist(df_csi_['delta_t_us'].dropna(), bins=50)
plt.xlabel('Delta time (us)')
plt.ylabel('Count')
plt.title('Histogram of Time Intervals Between CSI Samples')
plt.show()

Timestamp is periodic; Fast Fourier Transform provides the periodicity of timestamp; The wifi CSI data is possibly collected sequentially at periodic time interval leading to high importance of the time stamp feature on classification

In [ ]:
### Single dip in frequency transformation: suggests periodicity: use sin & cos features ##
delta_t = df_csi_['delta_t_us'].dropna()
fft = np.fft.fft(delta_t - np.mean(delta_t))
freq = np.fft.fftfreq(len(delta_t), d=1)

# Peak frequency: periodicity
mag = np.abs(fft)
mag[0] = 0
peak_idx = np.argmax(mag)
peak_freq = freq[peak_idx]
period = 1 / peak_freq if peak_freq != 0 else np.inf
print(f"Periodicity:{period}")

plt.plot(freq, np.abs(fft))
plt.xlabel('Frequency')
plt.ylabel('Magnitude')
plt.title('FFT of delta times')
plt.show()

In [ ]:
df_csi_['time_s'] = (df_csi_['timestamp'] / 1_000_000)   # seconds
df_csi_['time_rel_s'] = df_csi_['time_s'] - df_csi_['time_s'].iloc[0]
df_csi_['delta_t_us'] = df_csi_['time_s'].diff()
sns.pairplot(df_csi_[["time_rel_s","delta_t_us","position"]],hue='position');
plt.draw()

Timestamp box plot suggests that positions such as 0, 1 & 2 were largely measured across the time period wheres rest were measured closer to the initial time period.

In [ ]:
## Separation by time sequence
df_csi_.boxplot(column="time_rel_s", by= "position",figsize= (6,6))
plt.show()

### Amplitude Separability by Subcarriers & Mean
T-SNE, distribution, boxplot

In [ ]:
df_eda_amp_x, df_eda_amp_y, phase_df1, phase_df2, amp_df1, amp_df2 = generate_features(df_csi)
# keep_cols, drop_cols = fit_near_zero_variance(df_eda_amp_x, var_thresh=1e-4)
# df_eda_amp_x = df_eda_amp_x[keep_cols]
amp_df1 = pd.DataFrame(amp_df1)
amp_df2 = pd.DataFrame(amp_df2)

energy retained by 13 components > 99% - and has best CV result across folds

In [ ]:
dct_en = []
d1 = df_eda_amp_x.filter(regex=r"^dct_amp1").to_numpy()
denom = np.mean(np.sum(d1**2,axis=1))
for i in range(1,65):    
    d2 = df_eda_amp_x.filter(regex=r"^dct_amp1").iloc[:,:i].to_numpy()
    numer = np.mean(np.sum(d2**2,axis=1))
        
    # print(numer/denom)
    endc = numer/denom
    dct_en.append(endc)

plt.figure(figsize=(9, 2))
# With timestamp
plt.plot(
    [j for j in range(64)],
    dct_en,
    marker="o",
    linestyle="-"
)

plt.xlabel("Coefficients")
plt.ylabel("Cumulative Energy")
plt.title("Energy Retention for DCT Coefficients")
plt.legend(ncol=2)
plt.grid(True)
# plt.ylim(0, 1)
plt.show()

### T-SNE with all features ###
Demonstrates better separability and large clusters are more disctint with denoised data

In [ ]:
X = df_eda_amp_x.to_numpy()
y = df_eda_amp_y.to_numpy()
print(X.shape)
X_scaled = StandardScaler().fit_transform(X)

tsne = TSNE(
    n_components=2,
    perplexity=50,
    learning_rate=50,
    random_state=2
)

X_tsne = tsne.fit_transform(X_scaled)
plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, cmap='tab10', s=5)
plt.title("t-SNE after denoising")
plt.show()

In [ ]:
# ### RAW AMPLITUDE ### 
# amp_df1.columns = ["amp1"+str(i) for i in amp_df1.columns]
# amp_df2.columns = ["amp2"+str(i) for i in amp_df1.columns]
# df = pd.concat([amp_df1, amp_df1], axis=1)
# drop_dct = df.filter(regex=r"^dct").columns
# df = df.drop(columns=drop_dct)

# X = df.to_numpy()
# y = df_eda_amp_y.to_numpy()
# print(X.shape)
# X_scaled = StandardScaler().fit_transform(X)

# tsne = TSNE(
#     n_components=2,
#     perplexity=50,
#     learning_rate=50,
#     random_state=2
# )

# X_tsne = tsne.fit_transform(X_scaled)
# plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, cmap='tab10', s=5)
# plt.title("t-SNE visualization of class separability")
# plt.show()

### Check Separability with MDA

In [ ]:
# mda = LinearDiscriminantAnalysis(n_components=9)
# amp_df1_mda = pd.DataFrame(mda.fit_transform(amp_df1,df_eda_amp_y))
# amp_df2_mda = pd.DataFrame(mda.fit_transform(amp_df2,df_eda_amp_y))
# amp_df1_mda.columns = ["amp1"+str(i) for i in amp_df1_mda.columns]
# amp_df2_mda.columns = ["amp2"+str(i) for i in amp_df2_mda.columns]
# df = pd.concat([df_eda_amp_x, amp_df1_mda, amp_df2_mda], axis=1)
# drop_dct = df.filter(regex=r"^dct").columns
# df = df.drop(columns=drop_dct)

# X = df.to_numpy()
# y = df_eda_amp_y.to_numpy()
# print(X.shape)
# X_scaled = StandardScaler().fit_transform(X)

# tsne = TSNE(
#     n_components=2,
#     perplexity=50,
#     learning_rate=50,
#     random_state=2
# )

# X_tsne = tsne.fit_transform(X_scaled)
# plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, cmap='tab10', s=5)
# plt.title("t-SNE visualization of class separability")
# plt.show()

### Check Separability with PCA

In [ ]:
# amp_df1_pca = pd.DataFrame(PCA(10).fit_transform(amp_df1))
# amp_df2_pca = pd.DataFrame(PCA(10).fit_transform(amp_df2))
# amp_df1_pca.columns = ["amp1"+str(i) for i in amp_df1_pca.columns]
# amp_df2_pca.columns = ["amp2"+str(i) for i in amp_df2_pca.columns]
# df = pd.concat([df_eda_amp_x, amp_df1_pca, amp_df2_pca], axis=1)
# drop_dct = df.filter(regex=r"^dct").columns
# df = df.drop(columns=drop_dct)

# X = df.to_numpy()
# y = df_eda_amp_y.to_numpy()
# print(X.shape)
# X_scaled = StandardScaler().fit_transform(X)

# tsne = TSNE(
#     n_components=2,
#     perplexity=50,
#     learning_rate=50,
#     random_state=2
# )

# X_tsne = tsne.fit_transform(X_scaled)
# plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, cmap='tab10', s=5)
# plt.title("t-SNE visualization of class separability")
# plt.show()

### Plot Amplitude & Phase by Subcarriers

In [ ]:
plot_amplitude_per_subcarrier(amp_df1, df_eda_amp_y.values)
plot_amplitude_per_subcarrier(amp_df2, df_eda_amp_y.values)

In [ ]:
plot_amplitude_per_subcarrier(phase_df1, df_eda_amp_y.values)
plot_amplitude_per_subcarrier(phase_df2, df_eda_amp_y.values)

In [ ]:
plot_amplitude_distribution(amp_df1.to_numpy(), df_eda_amp_y.values)
plot_amplitude_distribution(amp_df2.to_numpy(), df_eda_amp_y.values)

In [ ]:
plot_amplitude_distribution(phase_df1.to_numpy(), df_eda_amp_y.values)
plot_amplitude_distribution(phase_df2.to_numpy(), df_eda_amp_y.values)

# MODEL TRAINING & VALIDATION

### Group K fold: periodicity from FFT
Groups train & validation set based on train groups to limit possibility of leakage due to time stamp

In [ ]:
## Feature - Target Split ##
X = df_csi.drop(columns="position")
y = df_csi["position"]

## Group K Fold to group based on similar time groups for validation
# TIME_BIN_US = 12_887  # binsize from FFT
TIME_BIN_US = 1_000_000 ## 1 s example
groups = (X["timestamp"] // TIME_BIN_US).astype(int)
gkf = GroupKFold(n_splits=5)

i=0
macro_metric = pd.DataFrame()
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    df_csi_train_X, df_csi_test_X = X.iloc[train_idx], X.iloc[val_idx]
    df_csi_train_y, df_csi_test_y = y.iloc[train_idx], y.iloc[val_idx]
    df_csi_train =  pd.concat([df_csi_train_X, df_csi_train_y],axis=1).reset_index(drop=True)
    df_csi_test =  pd.concat([df_csi_test_X, df_csi_test_y],axis=1).reset_index(drop=True)
    df_csi_train = df_csi_train.sort_values("timestamp").reset_index(drop=True)
    df_csi_test = df_csi_test.sort_values("timestamp").reset_index(drop=True)

    print(f"Train_len: {df_csi_train.shape[0]}, Test_len: {df_csi_test.shape[0]}")
    print(f"fold: {i}")
    i+=1
    metric_df, df_pred, test_x, train_x = model_run(df_csi_train,df_csi_test,"testing",63,23)
    macro_metric = pd.concat([macro_metric,metric_df])

print(f"Macro F1: {macro_metric["F1"].mean()}")

### K fold Validation

In [ ]:
## Feature - Target Split ##
X = df_csi.drop(columns="position")
y = df_csi["position"]

### K-FOLD validation check to randamize folds and test across ###
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=5)
df_errors = pd.DataFrame()

i=0
macro_metric = pd.DataFrame()
for train_idx, test_idx in kf.split(X,y):
    df_csi_train_X, df_csi_test_X = X.iloc[train_idx], X.iloc[test_idx]
    df_csi_train_y, df_csi_test_y = y.iloc[train_idx], y.iloc[test_idx]
    df_csi_train =  pd.concat([df_csi_train_X, df_csi_train_y],axis=1).reset_index(drop=True)
    df_csi_test =  pd.concat([df_csi_test_X, df_csi_test_y],axis=1).reset_index(drop=True)
    df_csi_train = df_csi_train.sort_values("timestamp").reset_index(drop=True)
    df_csi_test = df_csi_test.sort_values("timestamp").reset_index(drop=True)

    print(f"Train_len: {df_csi_train.shape[0]}, Test_len: {df_csi_test.shape[0]}")
    print(f"fold: {i}")
    i+=1
    metric_df, df_pred, test_x, train_x = model_run(df_csi_train,df_csi_test,"testing",63,3)
    macro_metric = pd.concat([macro_metric,metric_df])

print(f"Macro F1: {macro_metric["F1"].mean()}")

### Grid Search with KF Validation for hyperparameter tuning

In [ ]:
# param_grid = {
#         "clf__n_estimators":[600, 1200],
#         "clf__max_depth":[4,5,6],
#         "clf__learning_rate":[0.04,0.05],
#         "clf__reglambda":[1,1.5,2,4],
#         "clf__min_child_samples":[5,10],
# }

# X_train, y_train, phase_df1_train, phase_df2_train, amp1_train, amp2_train = generate_features(df_csi)

# grid_search_cv = GridSearchCV(estimator = xgboost.XGBClassifier(
#                                 objective="multi:softmax",
#                                 num_class=y_train.nunique(),
#                                 random_state=42,
#                                 eval_metric="mlogloss",
#                                 class_weight="balanced",
#                                 n_jobs=-1
#                             ),
#                             cv = KFold(n_splits=5, shuffle=False), param_grid=param_grid, scoring='f1', return_train_score=True)

# keep_cols, drop_cols = fit_near_zero_variance(X_train, var_thresh=1e-4)
# X_train = X_train[keep_cols]
# print(len(X_train.columns))
# grid_search_cv.fit(X_train, y_train)

# print(grid_search_cv.best_params_)

### SHAP ANALYSIS
Feature Importance & interpretation of feature impact on class prediction.\
DCT features are most important and based on specific important cosine coefficient component: specific ratio features are derived

In [ ]:
## Feature - Target Split ##
X = df_csi.drop(columns="position")
y = df_csi["position"]

### Prep Train - Test for SHAP analysis
df_csi_train_X, df_csi_test_X, df_csi_train_y, df_csi_test_y = train_test_split(
    X, y, train_size=0.7,
    random_state=42, shuffle=True, stratify=y
)

df_csi_train =  pd.concat([df_csi_train_X, df_csi_train_y],axis=1)
df_csi_test =  pd.concat([df_csi_test_X, df_csi_test_y],axis=1)
df_csi_train = df_csi_train.sort_values("timestamp").reset_index(drop=True)
df_csi_test = df_csi_test.sort_values("timestamp").reset_index(drop=True)
X_train, y_train, phase_df1_train, phase_df2_train, amp1_train, amp2_train = generate_features(df_csi_train)
X_val, y_val, phase_df1_test, phase_df2_test, amp1_test, amp2_test = generate_features(df_csi_test)

In [ ]:
## Shap Explainer ##
xgb_params = {
            "objective":"multi:softprob", 
            "num_class":y_train.nunique(),
            "n_estimators":600,
            "max_depth":5,
            "tree_method": "hist",
            "learning_rate":0.05,
            "subsample":0.8,
            "colsample_bytree":0.8,
            "random_state":42,
            "eval_metric":"mlogloss",
            "reg_lambda":1,
            "n_jobs":-1,
            "class_weight":"balanced",
            "min_child_samples":5
        }
    
xgb_mod =  xgboost.XGBClassifier(**xgb_params)
# keep_cols = ['time_rel_s', 'rssi_sum', 'rssi2', 'rssi1', 'dct_amp2_3', 'dct_amp1_4', 'dct_amp2_4', 'dct_energy1', 
#             'dct_amp2_7', 'dct_amp2_2', 'dct_amp1_11', 'dct_amp2_5', 'dct_amp2_9', 'dct_amp1_7', 'dct5_mean', 
#             'dct_ratio_4_5', 'dct_amp1_6', 'dct_amp1_12', 'dct_amp1_1', 'dct_amp2_6', 'dct4_mean', 'rssi_diff', 
#             'rssi_mean', 'dct_amp1_2', 'dct_amp2_8', 'dct_amp1_10', 'dct_amp2_11', 'dct_energy2', 'dct_amp1_8', 
#             'dct_amp1_5', 'dct_amp1_0', 'dct_amp1_9', 'dct6_mean', 'dct_ratio_3_4', 'dct_amp2_1', 'dct_amp1_3', 
#             'dct_amp2_0', 'dct_amp2_10', 'dct5_diff', 'time_cos', 'dct_amp2_12', 'rssi_mean_rolling', 'delta_t_s', 
#             'time_sin', 'dct_energy_diff', 'psd_max_diff', 'rssi2_rolling', 'rssi1_rolling', 'seq_ctrl_diff', 
#             'psd_max_ratio', 'phase_slope1', 'phase_diff_c', 'rssi_abs_diff', 'phase_diff_s', 'aoa_calculated2', 'psd1', 
#             'rssi_mean_sq', 'psd2', 'phase_slope2', 'p_slope_mean', 'tau_delay1', 'aoa_cos', 'tau_delay2']
# X_train = X_train[keep_cols]
# X_val  = X_val[keep_cols]
xgb_mod.fit(X_train, y_train)
explainer = shap.TreeExplainer(xgb_mod)

preds = xgb_mod.predict(X_val)
f1_score(y_val, preds, average="macro")

#### Identify the top most important features

In [ ]:
shap_values = explainer.shap_values(X_val)
if isinstance(shap_values, list):
    shap_abs_mean = np.mean([np.abs(sv) for sv in shap_values], axis=0)
else:
    shap_abs_mean = np.mean(np.abs(shap_values), axis=2)
shap.summary_plot(shap_abs_mean, X_val, plot_type="bar")

In [ ]:
### feature impact on prediction
shap.summary_plot(shap_abs_mean, X_val, max_display=6, show=True,plot_size=(7, 2))

#### Class wise top important features

In [ ]:
class_k = 4
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[class_k], X_val)
else:
    shap.summary_plot(shap_values[:, :, class_k], X_val)

# class_k = 2
# if isinstance(shap_values, list):
#     shap.summary_plot(shap_values[class_k], X_val)
# else:
#     shap.summary_plot(shap_values[:, :, class_k], X_val)

### Probability Prediction: CV bagging

In [ ]:
# df_csi_train_X, df_csi_test_X, df_csi_train_y, df_csi_test_y = train_test_split(
#     X, y, train_size=0.8,
#     random_state=42, shuffle=True, stratify=y
# )

# df_csi_train =  pd.concat([df_csi_train_X, df_csi_train_y],axis=1)
# df_csi_test =  pd.concat([df_csi_test_X, df_csi_test_y],axis=1)

# df_csi_train = df_csi_train.sort_values("timestamp").reset_index(drop=True)
# df_csi_test = df_csi_test.sort_values("timestamp").reset_index(drop=True)

# X_train, y_train, phase_df1_train, phase_df2_train, amp1_train, amp2_train = generate_features(df_csi_train)
# X_test, y_test, phase_df1_test, phase_df2_test, amp1_test, amp2_test = generate_features(df_csi_test)

# ## prune features: Dynamic selection ##
# keep_cols, drop_cols = fit_near_zero_variance(X_train, var_thresh=1e-4)
# X_train = X_train[keep_cols]
# X_test  = X_test[keep_cols]
# print(X_test.shape)

# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# oof_proba = np.zeros((X.shape[0], 10))
# test_proba = np.zeros((X_test.shape[0], 10))

# fold_scores = []

# for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train, y_train), 1):
#     X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
#     y_tr, y_val = y_train[tr_idx], y_train[val_idx]
#     # print(tr_idx,val_idx)

#     ### Initialize model Pipeline: XGBoost ###
#     xgb_mod = xgboost.XGBClassifier(
#             objective="multi:softprob",   # Correct objective for multiclass
#             num_class=y_train.nunique(),
#             n_estimators=600,
#             max_depth=5,
#             learning_rate=0.05,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             random_state=42,
#             eval_metric="mlogloss",
#             reg_lambda=4,
#             n_jobs=-1,
#             class_weight="balanced",
#             min_child_samples=5
#     )

#     xgb_mod.fit(
#         X_tr, y_tr,
#         eval_set=[(X_val, y_val)],
#         verbose=0
#     )

#     # OOF proba
#     val_proba = xgb_mod.predict_proba(X_val)
#     oof_proba[val_idx] = val_proba

#     # Fold score
#     val_pred = np.argmax(val_proba, axis=1)
#     fold_f1 = f1_score(y_val, val_pred, average="macro")
#     fold_scores.append(fold_f1)

#     # Test proba accumulation (bagging)
#     test_proba += xgb_mod.predict_proba(X_test) / 5

#     print(f"Fold {fold}: macro-F1 = {fold_f1:.5f}")
#     print(f"Test {fold}: macro-F1 = {f1_score(y_test, np.argmax(test_proba, axis=1), average="macro"):.5f}")

# # overall_f1 = f1_score(y_train, np.argmax(oof_proba, axis=1), average="macro")
# overall_f1_test = f1_score(y_test, np.argmax(test_proba, axis=1), average="macro")
# print(f"\nTest macro-F1: {overall_f1_test:.5f}")
# print(f"Fold F1 mean: {np.mean(fold_scores):.5f}  std: {np.std(fold_scores):.5f}")

### TSNE Check on Leaf samples from XGBoost

In [ ]:
# # predicted probability 
# idx = np.random.RandomState(42).choice(len(X_val), size=2000, replace=False)
# X_small = X_val.iloc[idx]
# y_small = y_val[idx]

# # proba_val = xgb_mod.predict_proba(X_small)   # works for RF too

# # tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto",
# #             init="pca", random_state=42)

# # Z = tsne.fit_transform(proba_val)
# # plt.figure(figsize=(7,6))
# # plt.scatter(Z[:,0], Z[:,1], c=y_small, s=12)
# # plt.title("t-SNE on model predicted probabilities")
# # plt.xlabel("t-SNE 1")
# # plt.ylabel("t-SNE 2")
# # plt.colorbar(label="True Class")
# # plt.show()


# # Leaf indices: shape (n_samples, n_trees)
# leaf_train = xgb_mod.apply(X_train)   # each value is leaf-id
# leaf_val   = xgb_mod.apply(X_small)
# leaf_val = leaf_val.astype(np.float32)

# tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto",
#             init="pca", random_state=42)

# Z = tsne.fit_transform(leaf_val)
# plt.figure(figsize=(7,6))
# plt.scatter(Z[:,0], Z[:,1], c=y_small, s=8)
# plt.title("t-SNE Cluster Purity Post Tuning")
# plt.xlabel("t-SNE 1")
# plt.ylabel("t-SNE 2")
# plt.colorbar(label="Class")
# plt.show()

In [ ]:
### Calibration Curves ###
# from sklearn.calibration import calibration_curve
# from sklearn.preprocessing import label_binarize

# # Predict probabilities
# y_proba = xgb_mod.predict_proba(X_val)

# # Binarize labels (one-vs-rest)
# classes = np.unique(y_val)
# y_val_bin = label_binarize(y_val, classes=classes)

# plt.figure()

# for i, cls in enumerate(classes):
#     frac_pos, mean_pred = calibration_curve(
#         y_val_bin[:, i],
#         y_proba[:, i],
#         n_bins=10,
#         strategy="uniform"
#     )
#     plt.plot(mean_pred, frac_pos, marker='o', alpha=0.4, label=f"{cls}")

# # Perfect calibration reference
# plt.plot([0, 1], [0, 1], linestyle='--')
# plt.legend(ncol=2)
# plt.xlabel("Mean predicted probability")
# plt.ylabel("Fraction of positives")
# plt.title("Calibration curves (Pre-tuning)")
# plt.show()

# BLIND PREDICTION ON TEST DATA

In [ ]:
## Initialize train data
df_csi_train =  df_csi.reset_index(drop=True)
df_csi_test = df_csi_test.sort_values("timestamp").reset_index(drop=True)

## Read Test Data
df_csi_test = pd.read_csv(paths["test_dataset"])
indexes = df_csi_test['ID']
df_csi_test['position']=0
df_csi_test = df_csi_test.sort_values("timestamp").reset_index(drop=True)
df_csi_test

In [ ]:
### MODEL RUN & PREDICTION
print(f"Train_len: {df_csi_train.shape[0]}, Test_len: {df_csi_test.shape[0]}")
f1_score_, df_pred, test_x, train_x = model_run(df_csi_train,df_csi_test,"blind",63,23)
df_pred = df_pred.sort_values(by="ID").reset_index(drop=True)
df_pred

In [ ]:
### Write the test data results ###
df_pred.to_csv(paths["output_dataset"], index=False)